In [20]:
import apache_beam as beam
import json
import importlib
import os
import unittest
import pandas as pd

from apache_beam.transforms.util import WaitOn
from apache_beam.options.pipeline_options import PipelineOptions
from datetime import datetime

In [21]:
fromNotebook = True
configFile = "warehouse_wwi_2014-01-01.json"
loadConfigFile = "load_wwi_2014-01-01.json"
newCutoffDate = "2014-01-01"
dimension_tables = []
fact_tables = []

In [22]:
prefix = ""
if fromNotebook == False:
    prefix = "notebooks/"

file_path = os.path.join(os.getcwd(), f"{prefix}modules", 'sql_utilities.py')
spec = importlib.util.spec_from_file_location("sql_utils", file_path)
sql_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sql_utils)

In [23]:
f = open(loadConfigFile,)
source_config = json.load(f)
f.close()

load_tables = [
    {
        "name": table["name"],
        "database": source_config["destination"]["database"],
        "schema": table["destination"]["schema"],
        "table": table["destination"]["table"]
    }
    for table in source_config["tables"]
]

print(f"load_tables: {load_tables}")

load_tables: [{'name': 'ApplicationCities', 'database': 'WideWorldImportersDW', 'schema': 'dbo', 'table': 'Application_Cities'}, {'name': 'ApplicationCitiesArchive', 'database': 'WideWorldImportersDW', 'schema': 'dbo', 'table': 'Application_Cities_Archive'}, {'name': 'ApplicationCountries', 'database': 'WideWorldImportersDW', 'schema': 'dbo', 'table': 'Application_Countries'}, {'name': 'ApplicationCountriesArchive', 'database': 'WideWorldImportersDW', 'schema': 'dbo', 'table': 'Application_Countries_Archive'}, {'name': 'ApplicationDeliveryMethods', 'database': 'WideWorldImportersDW', 'schema': 'dbo', 'table': 'Application_DeliveryMethods'}, {'name': 'ApplicationDeliveryMethodsArchive', 'database': 'WideWorldImportersDW', 'schema': 'dbo', 'table': 'Application_DeliveryMethods_Archive'}, {'name': 'ApplicationPaymentMethods', 'database': 'WideWorldImportersDW', 'schema': 'dbo', 'table': 'Application_PaymentMethods'}, {'name': 'ApplicationPaymentMethodsArchive', 'database': 'WideWorldImpor

In [24]:
f = open(configFile,)
config = json.load(f)
f.close()

source_db = config["source"]
destination_db = config["destination"]
initialCutoffDate = config["initialCutoffDate"]
if fromNotebook == True:
    dimension_tables = config["dimensionTables"]
    fact_tables = config["factTables"]
    newCutoffDate = config["cutoffDate"]
lastCutoffDate = "2012-12-31"
warehouse_tables = (
    [
        {
            "name": table["name"],
            "database": config["destination"]["database"],
            "schema": table["schema"],
            "table": table["table"]
        }
        for table in config["dimensionTables"]
    ] + 
    [
        {
            "name": table["name"],
            "database": config["destination"]["database"],
            "schema": table["schema"],
            "table": table["table"]
        }
        for table in config["factTables"]
    ]
)
spark_master = config["sparkMaster"]
spark_jars = config["sparkJars"]
spark_executor_memory = config["sparkExecutorMemory"]
spark_driver_memory = config["sparkDriverMemory"]
spark_executor_cores = config["sparkExecutorCores"]
spark_cores_max = config["sparkCoresMax"]
spark_network_timeout = config["sparkNetworkTimeout"]
spark_executor_heartbeat_interval = config["sparkExecutorHeartBeatInterval"]
runner_options = config["runnerOptions"][config["runner"]]
gcp_credential = config["gcpCredential"]

print(f"source_db: {source_db}")
print(f"destination_db: {destination_db}")
print(f"dimension_tables: {dimension_tables}")
print(f"fact_tables: {fact_tables}")
print(f"initialCutoffDate: {initialCutoffDate}")
print(f"newCutoffDate: {newCutoffDate}")
print(f"lastCutoffDate: {lastCutoffDate}")
print(f"warehouse_tables: {warehouse_tables}")
print(f"spark_master: {spark_master}")
print(f"spark_jars: {spark_jars}")
print(f"spark_executor_memory: {spark_executor_memory}")
print(f"spark_driver_memory: {spark_driver_memory}")
print(f"spark_executor_cores: {spark_executor_cores}")
print(f"spark_cores_max: {spark_cores_max}")
print(f"spark_network_timeout: {spark_network_timeout}")
print(f"spark_executor_heartbeat_interval: {spark_executor_heartbeat_interval}")
print(f"runner_options: {runner_options}")
print(f"gcp_credential: {gcp_credential}")

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = gcp_credential

source_db: {'databaseType': 'mssql', 'database': 'WideWorldImportersDW', 'conn': 'DRIVER={ODBC Driver 17 for SQL Server};SERVER=localhost,1433;DATABASE=WideWorldImportersDW;UID=sa;PWD=PP@$$w0rd', 'conn_spark': 'jdbc:sqlserver://localhost;database=WideWorldImportersDW;user=sa;password=PP@$$w0rd;encrypt=false'}
destination_db: {'databaseType': 'mssql', 'database': 'WideWorldImportersDW', 'conn': 'DRIVER={ODBC Driver 17 for SQL Server};SERVER=localhost,1433;DATABASE=WideWorldImportersDW;UID=sa;PWD=PP@$$w0rd', 'conn_spark': 'jdbc:sqlserver://localhost;database=WideWorldImportersDW;user=sa;password=PP@$$w0rd;encrypt=false'}
dimension_tables: [{'name': 'DimCities', 'schema': 'dbo', 'table': 'DimCities', 'createTable': 'scripts/warehouse_wwi_mssql/create_dimension_cities.sql', 'upsertTable': 'scripts/warehouse_wwi_mssql/upsert_dimension_cities.sql', 'specificColumnTypes': [{'name': 'Location', 'type': 'VARBINARY(MAX)', 'typeCast': 'CAST(? AS VARBINARY(MAX))'}], 'primaryKey': 'CityKey', 'modif

In [25]:
if source_db["databaseType"].startswith("spark-") and destination_db["databaseType"].startswith("spark-"):
    sql_utils.initialize_spark_session(
        master=spark_master,
        jdbc_url=source_db["conn"], 
        jars=spark_jars, 
        executor_memory=spark_executor_memory, 
        driver_memory=spark_driver_memory, 
        executor_cores=spark_executor_cores, 
        cores_max=spark_cores_max, 
        network_timeout=spark_network_timeout, 
        executor_heartbeat_interval=spark_executor_heartbeat_interval
    )

In [26]:
def get_last_cutoff_date(table, last_cutoff_dates):
    last_cutoff_date = next((item for item in last_cutoff_dates if item.get("TableName") == table), None)

    if last_cutoff_date is None:
        return initialCutoffDate
    else:
        return last_cutoff_date["LastCutoffDate"].strftime("%Y-%m-%d %H:%M:%S")

In [27]:
def upsert_data_base(conn_str, database_type, database, upsert_path, values_to_replace, tables_to_replace, show_sql = False):
    sql = sql_utils.get_sql_from_script(
        upsert_path, 
        values_to_replace, 
        tables_to_replace,
        database_type
    )

    if show_sql == True:
        print(f"""Upsert Data sql:
                {sql} 
                """)
    
    sql_utils.exec_sql(conn_str, sql, None, database_type, database, spark_master, spark_jars)

In [28]:
class UpsertData(beam.DoFn):
    def __init__(self, database_type, conn_str, upsert_path, values_to_replace, tables_to_replace, yield_record = None, show_sql = False, database = ""):
        self.database_type = database_type
        self.conn_str = conn_str
        self.upsert_path = upsert_path
        self.values_to_replace = values_to_replace
        self.tables_to_replace = tables_to_replace
        self.yield_record = yield_record
        self.show_sql = show_sql
        self.database = database

    def process(self, element):
        upsert_data_base(
            self.conn_str, 
            self.database_type, 
            self.database, 
            self.upsert_path, 
            self.values_to_replace, 
            self.tables_to_replace, 
            self.show_sql
        )

        if self.yield_record is not None:
            yield self.yield_record

In [29]:
pd.set_option('display.max_colwidth', None)
tc = unittest.TestCase()

class WarehouseData(beam.DoFn):
    def __init__(self, upsert_config, assert_config, insert_history_config, modify_table_config = None, yield_record = None):
        self.upsert_config = upsert_config
        self.assert_config = assert_config
        self.insert_history_config = insert_history_config
        self.modify_table_config = modify_table_config
        self.yield_record = yield_record
    
    def upsert_data(self):
        upsert_data_base(
            self.upsert_config["conn"],
            self.upsert_config["database_type"], 
            self.upsert_config["database"], 
            self.upsert_config["path"], 
            self.upsert_config["values_to_replace"], 
            self.upsert_config["tables_to_replace"], 
            self.upsert_config["show_sql"]
        )

    def assert_data(self):
        if self.assert_config["test"] == True:
            error_counts = []

            if len(self.assert_config["unit_tests"]) > 0:
                count_sqls = []
                
                for unit_test in self.assert_config["unit_tests"]:
                    common_values = [
                        { "name": "Name", "value": unit_test["name"] },
                        { "name": "Column", "value": unit_test["column"] },
                        { "name": "Type", "value": unit_test["type"] },
                        { "name": "SubType", "value": unit_test["subType"] }
                    ]

                    if unit_test["type"] == "column":
                        if unit_test["subType"] == "relationship":
                            count_sqls.append(
                                sql_utils.get_sql_from_script(
                                    self.assert_config["test_config"][unit_test["subType"]],
                                    self.assert_config["values_to_replace"] + common_values + [
                                        { "name": "ParentColumn", "value": unit_test["parentColumn"] },
                                        { "name": "ParentTable", "value": unit_test["parentTable"] }
                                    ], 
                                    self.assert_config["tables_to_replace"],
                                    self.assert_config["database_type"]
                                ) 
                            )
                        elif unit_test["subType"] == "expressionTrue":
                            count_sqls.append(
                                sql_utils.get_sql_from_script(
                                    self.assert_config["test_config"][unit_test["subType"]],
                                    self.assert_config["values_to_replace"] + common_values + [
                                        { "name": "Expression", "value": unit_test["expression"] }
                                    ], 
                                    self.assert_config["tables_to_replace"],
                                    self.assert_config["database_type"]
                                ) 
                            )
                        else:
                            count_sqls.append(
                                sql_utils.get_sql_from_script(
                                    self.assert_config["test_config"][unit_test["subType"]],
                                    self.assert_config["values_to_replace"] + common_values, 
                                    self.assert_config["tables_to_replace"],
                                    self.assert_config["database_type"]
                                ) 
                            )
                    elif unit_test["type"] == "table":
                        count_sqls.append(
                            sql_utils.get_sql_from_script(
                                unit_test["testScript"],
                                self.assert_config["values_to_replace"] + common_values, 
                                self.assert_config["tables_to_replace"],
                                self.assert_config["database_type"]
                            ) 
                        )

                if len(count_sqls) > 0:
                    count_sql = sql_utils.get_sql_from_script(
                        self.assert_config["test_config"]["selectCounts"],
                        [
                            {
                                "name": "SelectCountsSql",
                                "value": " UNION ALL ".join(count_sqls)
                            }
                        ],
                        self.assert_config["tables_to_replace"],
                        self.assert_config["database_type"]
                    )

                    if self.assert_config["show_sql"] == True:
                        print(f"count_sql: {count_sql}",)

                    error_counts = list(
                        sql_utils.select_sql(
                            self.assert_config["conn"], 
                            count_sql, 
                            self.assert_config["database_type"],
                            "dictionary",
                            self.assert_config["database"],
                            spark_jars,
                            spark_master,
                            False
                        )
                    )
                    
            if len(error_counts) > 0:
                print("Error occurred on the following tables:")
                
                display(pd.DataFrame(error_counts))

                raise("Assert data error.  Please see error above.")
            
    def insert_history_record(self):
        upsert_data_base(
            self.insert_history_config["conn"],
            self.insert_history_config["database_type"], 
            self.insert_history_config["database"], 
            self.insert_history_config["path"], 
            self.insert_history_config["values_to_replace"], 
            self.insert_history_config["tables_to_replace"], 
            self.insert_history_config["show_sql"]
        )

    def modify_table(self):
        upsert_data_base(
            self.modify_table_config["conn"],
            self.modify_table_config["database_type"], 
            self.modify_table_config["database"], 
            self.modify_table_config["path"], 
            self.modify_table_config["values_to_replace"], 
            self.modify_table_config["tables_to_replace"], 
            self.modify_table_config["show_sql"]
        )

    def process(self, element):
        # Modify Table
        if self.modify_table_config is not None:
            self.modify_table()

        # Upsert Data
        self.upsert_data()

        # Assert Data
        self.assert_data()

        # Insert History Record
        self.insert_history_record()

        if self.yield_record is not None:
            yield self.yield_record

In [30]:
class LoadData(beam.DoFn):
    def __init__(self, database_type, conn_str, load_path, values_to_replace, tables_to_replace, show_sql = False, result_type = "tuple", set_timestamp_tostring = False):
        self.database_type = database_type
        self.conn_str = conn_str
        self.load_path = load_path
        self.values_to_replace = values_to_replace
        self.tables_to_replace = tables_to_replace
        self.show_sql = show_sql
        self.result_type = result_type
        self.set_timestamp_tostring = set_timestamp_tostring

    def process(self, element):
        sql = sql_utils.get_sql_from_script(
            self.load_path, 
            self.values_to_replace, 
            self.tables_to_replace
        )

        if self.show_sql == True:
            print(f"LoadData sql: {sql}")
        
        return sql_utils.select_sql(self.conn_str, sql, self.database_type, self.result_type, "", spark_jars, spark_master, self.set_timestamp_tostring)

In [31]:
class CreateTables(beam.DoFn):
    def __init__(self, database_type, conn_str, sql, database, yield_record = None, show_sql = False):
        self.database_type = database_type
        self.conn_str = conn_str
        self.sql = sql
        self.database = database
        self.yield_record = yield_record
        self.show_sql = show_sql

    def process(self, element):
        if len(self.sql) > 0:
            
            if self.show_sql == True:
                print(f"""Upsert Data sql:
                    {self.sql} 
                    """)
                
                print(f"element: {element}")
                print(f"len element: {len(element)}")

            sql_utils.exec_sql(self.conn_str, self.sql, None, self.database_type, self.database, spark_master, spark_jars)

        if self.yield_record is not None:
            yield self.yield_record

In [32]:
def get_last_cutoff_dates():
    sql = sql_utils.get_sql_from_script(
        config["warehouseHistory"]["loadLastCutoffDatesTable"], 
        [
            { "name": "Schema", "value": config["warehouseHistory"]["schema"] },
            { "name": "Table", "value": config["warehouseHistory"]["table"] },
            { "name": "Database", "value": destination_db["database"] }
        ], 
        []
    )

    return sql_utils.select_sql(
        destination_db["conn"], 
        sql, 
        destination_db["databaseType"], 
        "dictionary", 
        destination_db["database"], 
        spark_jars, 
        spark_master, 
        False, 
        False,
        spark_load_sql = True
    )

In [33]:
def get_create_tables_sql(tables, last_cutoff_dates, new_cutoff_date):
    create_table_sqls = []

    for table in tables:
        tableName = table["table"]
        last_cutoff_date = get_last_cutoff_date(tableName, last_cutoff_dates)
        lastCutoffDateVal = datetime.strptime(last_cutoff_date, "%Y-%m-%d %H:%M:%S")
        newCutoffDateVal = datetime.strptime(new_cutoff_date, "%Y-%m-%d %H:%M:%S")

        if lastCutoffDateVal < newCutoffDateVal:
            create_table_sqls.append(
                sql_utils.get_sql_from_script(
                    table["createTable"], 
                    [
                        { "name": "Database", "value": destination_db["database"] },
                        { "name": "Schema", "value": table["schema"] },
                        { "name": "Table", "value": table["table"] }
                    ], 
                    []
                )
                
            )

    if len(create_table_sqls) > 0:
        return " ".join(create_table_sqls)
    
    return ""

In [35]:
last_cutoff_dates = get_last_cutoff_dates()
print(f"last_cutoff_dates: {last_cutoff_dates}")

options = PipelineOptions(runner_options)

with beam.Pipeline(options=options) as p:
    def modify_table(table, process_table, last_cutoff_dates, wait_on_process):
        tableName = table["table"]
        last_cutoff_date = get_last_cutoff_date(tableName, last_cutoff_dates)
        modifyTableScripts = []
                
        for modifyTableScript in table["modifyTable"]:
            modifyTableScripts.append(
                sql_utils.get_sql_from_script(
                    config["modifyWarehouseHistory"]["loadUnprocessedModifyWarehouseHistoryScriptTable"],
                    [
                        { "name": "ScriptName", "value": modifyTableScript["tableUpdate"] },
                        { "name": "LoadScriptName", "value": modifyTableScript["tableDataLoad"] },
                        { "name": "UpdateScriptName", "value": modifyTableScript["tableDataUpdate"] },
                        { "name": "Name", "value": modifyTableScript["name"] }
                    ],
                    [],
                    destination_db["databaseType"]
                )
            )
        
        unprocessedScripts = sql_utils.select_spark_sql(
            destination_db["conn"],
            sql_utils.get_sql_from_script(
                config["modifyWarehouseHistory"]["loadUnprocessedModifyWarehouseHistoryTable"],
                [
                    { "name": "Database", "value": destination_db["database"] },
                    { "name": "Schema", "value": config["modifyWarehouseHistory"]["schema"] },
                    { "name": "Table", "value": config["modifyWarehouseHistory"]["table"] },
                    { "name": "TableName", "value": tableName },
                    { "name": "LoadDate", "value": last_cutoff_date },
                    { "name": "MHSchema", "value": config["warehouseHistory"]["schema"] },
                    { "name": "MHTable", "value": config["warehouseHistory"]["table"] },
                    { "name": "ModifyTableScripts", "value": " UNION ".join(modifyTableScripts) }
                ],
                [],
                destination_db["databaseType"]
            ),
            destination_db["databaseType"],
            destination_db["database"],
            spark_master,
            spark_jars
        )

        for script in unprocessedScripts:
            print(f"Processing '{script["Name"]}' for table {tableName} for cutoff {newCutoffDate}.")
            
            sql_utils.exec_sql(
                destination_db["conn"],
                sql_utils.get_sql_from_script(
                    script["ScriptName"], 
                    [
                        { "name": "Database", "value": destination_db["database"] },
                        { "name": "Schema", "value": table["schema"] },
                        { "name": "Table", "value": table["table"] },
                    ],
                    [],
                    destination_db["databaseType"]
                ),
                None,
                destination_db["databaseType"],
                destination_db["database"]
            )

            process_table = (
                process_table
                | f"Modify Table {tableName}" >> 
                    beam.ParDo(
                        WarehouseData(
                            modify_table_config={
                                "conn": destination_db["conn"],
                                "database_type": destination_db["databaseType"],
                                "database": destination_db["database"],
                                "path": script["ScriptName"],
                                "values_to_replace": [
                                    { "name": "Database", "value": destination_db["database"] },
                                    { "name": "Schema", "value": table["schema"] },
                                    { "name": "Table", "value": table["table"] },
                                ],
                                "tables_to_replace": [],
                                "show_sql": False
                            },
                            upsert_config={
                                "conn": destination_db["conn"],
                                "database_type": destination_db["databaseType"],
                                "database": destination_db["database"],
                                "path": script["UpdateScriptName"],
                                "values_to_replace": [
                                    { "name": "NewCutoffDate", "value": newCutoffDate },
                                    { "name": "LastCutoffDate", "value": last_cutoff_date }
                                ],
                                "tables_to_replace": load_tables + warehouse_tables,
                                "show_sql": False
                            },
                            assert_config={
                                "conn": destination_db["conn"],
                                "database_type": destination_db["databaseType"],
                                "database": destination_db["database"],
                                "test_config": config["testConfig"],
                                "unit_tests": modifyTableScript["unitTests"],
                                "values_to_replace": [
                                    { "name": "Database", "value": destination_db["database"] },
                                    { "name": "Schema", "value": table["schema"] },
                                    { "name": "Table", "value": table["table"] },
                                    { "name": "LoadDate", "value": last_cutoff_date },
                                    { "name": "LastCutoffDate", "value": last_cutoff_date },
                                    { "name": "NewCutoffDate", "value": newCutoffDate }
                                ],
                                "tables_to_replace": load_tables + warehouse_tables,
                                "test": config["test"],
                                "test_record_count": config["testRecordCount"],
                                "show_sql": False
                            },
                            insert_history_config={
                                "conn": destination_db["conn"],
                                "database_type": destination_db["databaseType"],
                                "database": destination_db["database"],
                                "path": config["modifyWarehouseHistory"]["insertModifyWarehouseHistoryTable"],
                                "values_to_replace": [
                                    { "name": "Database", "value": destination_db["database"] },
                                    { "name": "Schema", "value": config["modifyWarehouseHistory"]["schema"] },
                                    { "name": "Table", "value": config["modifyWarehouseHistory"]["table"] },
                                    { "name": "TableName", "value": tableName },
                                    { "name": "LoadDate", "value": last_cutoff_date },
                                    { "name": "Status", "value": "Successful" },
                                    { "name": "Details", "value": "" },
                                    { "name": "ScriptName", "value": script["Name"] }
                                ],
                                "tables_to_replace": [],
                                "show_sql": False
                            },
                            yield_record={ f"Successfully processed table {tableName} for {newCutoffDate}." }
                        )
                    )
                | f"Modify WaitOn Table {tableName} 4" >> WaitOn(*wait_on_process)
            )
        
        return process_table

    def get_process_table(table, wait_on_process, last_cutoff_dates):
        tableName = table["name"]

        print(f"Processing table {tableName} for {newCutoffDate}.")

        last_cutoff_date = get_last_cutoff_date(tableName, last_cutoff_dates)
        lastCutoffDateVal = datetime.strptime(last_cutoff_date, "%Y-%m-%d %H:%M:%S")
        newCutoffDateVal = datetime.strptime(newCutoffDate, "%Y-%m-%d %H:%M:%S")
        
        if lastCutoffDateVal >= newCutoffDateVal:
            print(f"Table '{tableName}' has already been loaded for '{newCutoffDate}'.")
        else:

            process_table = p | f"Input Create Table for {tableName}" >> beam.Create([{ "PlaceHolder": "Temp 1 record" }])

            if len(table["modifyTable"]) > 0:
                process_table = modify_table(table, process_table, last_cutoff_dates, [create_tables])

            return (
                process_table
                | f"Warehouse Table {tableName}" >> 
                    beam.ParDo(
                        WarehouseData(
                            upsert_config={
                                "conn": destination_db["conn"],
                                "database_type": destination_db["databaseType"],
                                "database": destination_db["database"],
                                "path": table["upsertTable"],
                                "values_to_replace": [
                                    { "name": "NewCutoffDate", "value": newCutoffDate },
                                    { "name": "LastCutoffDate", "value": last_cutoff_date }
                                ],
                                "tables_to_replace": load_tables + warehouse_tables,
                                "show_sql": False
                            },
                            assert_config={
                                "conn": destination_db["conn"],
                                "database_type": destination_db["databaseType"],
                                "database": destination_db["database"],
                                "test_config": config["testConfig"],
                                "unit_tests": table["unitTests"],
                                "values_to_replace": [
                                    { "name": "Database", "value": destination_db["database"] },
                                    { "name": "Schema", "value": table["schema"] },
                                    { "name": "Table", "value": table["table"] },
                                    { "name": "LoadDate", "value": newCutoffDate },
                                    { "name": "LastCutoffDate", "value": last_cutoff_date },
                                    { "name": "NewCutoffDate", "value": newCutoffDate }
                                ],
                                "tables_to_replace": load_tables + warehouse_tables,
                                "test": config["test"],
                                "test_record_count": config["testRecordCount"],
                                "show_sql": False
                            },
                            insert_history_config={
                                "conn": destination_db["conn"],
                                "database_type": destination_db["databaseType"],
                                "database": destination_db["database"],
                                "path": config["warehouseHistory"]["insertWarehouseHistoryTable"],
                                "values_to_replace": [
                                    { "name": "Database", "value": destination_db["database"] },
                                    { "name": "Schema", "value": config["warehouseHistory"]["schema"] },
                                    { "name": "Table", "value": config["warehouseHistory"]["table"] },
                                    { "name": "TableName", "value": tableName },
                                    { "name": "LoadDate", "value": newCutoffDate },
                                    { "name": "Status", "value": "Successful" },
                                    { "name": "Details", "value": "" }
                                ],
                                "tables_to_replace": [],
                                "show_sql": False
                            },
                            yield_record={ f"Successfully processed table {tableName} for {newCutoffDate}." }
                        )
                    )
                | f"Warehouse Table WaitOn {tableName}" >> WaitOn(*wait_on_process)
                | f"Result for Table {tableName}" >> beam.Map(print)
            )
            
    # Create warehouse history table if not existing
    create_warehouse_history_table = (
        p
        | "Input Create Warehouse History Table" >> beam.Create([{ "PlaceHolder": "Temp 1 record" }])
        | "Create Warehouse History Table" >> 
            beam.ParDo(
                UpsertData(
                    destination_db["databaseType"],
                    destination_db["conn"],
                    config["warehouseHistory"]["createWarehouseHistoryTable"],
                    [
                        { "name": "Database", "value": config["destination"]["database"] },
                        { "name": "Schema", "value": config["warehouseHistory"]["schema"] },
                        { "name": "Table", "value": config["warehouseHistory"]["table"] }
                    ],
                    [],
                    { "Successfully created warehouse history table." },
                    False,
                    destination_db["database"]
                )
            )
        | "Print Create Warehouse History Table" >> beam.Map(print)
    )

    # Create modify warehouse history table if not existing
    create_modify_warehouse_history_table = (
        p
        | "Input Create Modify Warehouse History Table" >> beam.Create([{ "PlaceHolder": "Temp 1 record" }])
        | "Create Modify Warehouse History Table" >>
            beam.ParDo(
                UpsertData(
                    destination_db["databaseType"],
                    destination_db["conn"],
                    config["modifyWarehouseHistory"]["createModifyWarehouseHistoryTable"],
                    [
                        { "name": "Database", "value": config["destination"]["database"] },
                        { "name": "Schema", "value": config["modifyWarehouseHistory"]["schema"] },
                        { "name": "Table", "value": config["modifyWarehouseHistory"]["table"] }
                    ],
                    [],
                    { "Successfully created modify warehouse history table." },
                    False,
                    destination_db["database"]
                )
            )
        | "Print Create Modify Warehouse History Table" >> beam.Map(print)
    )

    create_tables_sql = get_create_tables_sql(fact_tables + dimension_tables, last_cutoff_dates, newCutoffDate)
    create_tables = (
        p
        | "Input Create Tables" >> beam.Create([{ "PlaceHolder": "Temp 1 record" }])
        | "Create Tables" >> beam.ParDo(
            CreateTables(
                destination_db["databaseType"],
                destination_db["conn"],
                create_tables_sql,
                destination_db["database"],
                { f"Successfully created tables." },
                False
            )
        )
        | "Wait On History Tables" >> WaitOn(create_warehouse_history_table, create_modify_warehouse_history_table)
        | "Print Create Tables" >> beam.Map(print)
    )

    dimension_tables_processes = []
    if len(dimension_tables) > 0:
        for table in dimension_tables:    
            dimension_table_process = get_process_table(table, [create_tables], last_cutoff_dates)
            if dimension_table_process is not None:
                dimension_tables_processes.append(dimension_table_process)
        
    if len(fact_tables) > 0:
        if len(dimension_tables_processes) > 0:
            for table in fact_tables:
                get_process_table(table, dimension_tables_processes, last_cutoff_dates)
        else:
            for table in fact_tables:
                get_process_table(table, [create_tables], last_cutoff_dates)

last_cutoff_dates: [{'TableName': 'DimCities', 'LastCutoffDate': datetime.datetime(2014, 1, 1, 0, 0)}, {'TableName': 'DimCustomers', 'LastCutoffDate': datetime.datetime(2014, 1, 1, 0, 0)}, {'TableName': 'DimDates', 'LastCutoffDate': datetime.datetime(2014, 1, 1, 0, 0)}, {'TableName': 'DimEmployees', 'LastCutoffDate': datetime.datetime(2014, 1, 1, 0, 0)}, {'TableName': 'DimPaymentMethods', 'LastCutoffDate': datetime.datetime(2014, 1, 1, 0, 0)}, {'TableName': 'DimStockItems', 'LastCutoffDate': datetime.datetime(2014, 1, 1, 0, 0)}, {'TableName': 'DimSuppliers', 'LastCutoffDate': datetime.datetime(2014, 1, 1, 0, 0)}, {'TableName': 'DimTransactionTypes', 'LastCutoffDate': datetime.datetime(2014, 1, 1, 0, 0)}, {'TableName': 'FctMovements', 'LastCutoffDate': datetime.datetime(2014, 1, 1, 0, 0)}, {'TableName': 'FctOrders', 'LastCutoffDate': datetime.datetime(2014, 1, 1, 0, 0)}, {'TableName': 'FctPurchases', 'LastCutoffDate': datetime.datetime(2014, 1, 1, 0, 0)}, {'TableName': 'FctSales', 'Last